วิเคราะห์ Real Bullish Reversal

In [8]:
import yfinance as yf
import pandas as pd
import datetime

def calculate_rvol_30m(tickers, period='15d'):
    results = []
    
    for ticker in tickers:
        print(f"กำลังดึงข้อมูลและคำนวณของหุ้น: {ticker}...")
        try:
            # ดึงข้อมูลกราฟราย 15 นาที ย้อนหลัง
            df = yf.download(ticker, period=period, interval='15m', progress=False)
            
            if df.empty:
                print(f"  -> ไม่พบข้อมูลของ {ticker} (อาจโดนถอดออกจากตลาด หรือเปลี่ยนชื่อ)")
                continue

            # จัดการเรื่อง Timezone ให้เป็น US/Eastern (เวลาตลาดหุ้นอเมริกา)
            df = df.reset_index()
            time_col = df.columns[0]
            
            if df[time_col].dt.tz is None:
                df[time_col] = df[time_col].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
            else:
                df[time_col] = df[time_col].dt.tz_convert('US/Eastern')

            # กรองเอาเฉพาะช่วง 09:30 ถึง 09:59
            df['Time'] = df[time_col].dt.time
            df_30m = df[(df['Time'] >= datetime.time(9, 30)) & (df['Time'] < datetime.time(11, 0))].copy()
            
            df_30m['Date'] = df_30m[time_col].dt.date
            
            # --- จุดที่แก้ไขแล้ว (Fix Pandas Tuple Error) ---
            if isinstance(df_30m.columns, pd.MultiIndex):
                # เข้าถึงคอลัมน์ Volume ชั้นแรกก่อน
                vol_data = df_30m['Volume']
                # ถ้าเป็น DataFrame แสดงว่ามีหลายคอลัมน์ ให้เลือกคอลัมน์ที่เป็นชื่อหุ้น
                if isinstance(vol_data, pd.DataFrame) and ticker in vol_data.columns:
                    target_vol = vol_data[ticker]
                else:
                    # ถ้าหาชื่อหุ้นไม่เจอ หรือมีแค่คอลัมน์เดียว ให้ดึงคอลัมน์แรกมาใช้เลย
                    target_vol = vol_data.iloc[:, 0]
            else:
                target_vol = df_30m['Volume']
                
            # เอา Series ที่ดึงมาได้ใส่กลับไปเป็นคอลัมน์ปกติ
            df_30m['Target_Volume'] = target_vol
            vol_series = df_30m.groupby('Date')['Target_Volume'].sum()
            # -----------------------------------------------
                
            if len(vol_series) < 2:
                print(f"  -> ข้อมูลของ {ticker} มีจำนวนวันไม่พอที่จะหาค่าเฉลี่ย")
                continue
                
            # แยกวันที่ล่าสุด (Today) ออกจากวันก่อนหน้า (Past Days)
            dates = sorted(vol_series.index)
            today = dates[-1]
            past_dates = dates[:-1]
            
            today_vol = float(vol_series[today])
            avg_past_vol = float(vol_series[past_dates].mean())
            
            # คำนวณ RVOL (Relative Volume)
            rvol = today_vol / avg_past_vol if avg_past_vol > 0 else 0
            
            # สรุปและประเมินสัญญาณ
            if rvol >= 1.5:
                signal = 'Strong Reversal (กลับตัวจริง)'
            elif 0.8 <= rvol < 1.5:
                signal = 'Normal / Sideways (ปกติ)'
            else:
                signal = 'Bearish Continuation (ลงต่อ)'
                
            results.append({
                'Ticker': ticker,
                'Date': today.strftime('%Y-%m-%d'),
                'Volume_Today (30m)': f"{int(today_vol):,}",
                'Avg_Volume (30m)': f"{int(avg_past_vol):,}",
                'RVOL': round(rvol, 2),
                'Signal': signal
            })
            
        except Exception as e:
            print(f"  -> เกิดข้อผิดพลาดกับ {ticker}: {e}")

    # แสดงผลออกมาเป็นตาราง DataFrame
    results_df = pd.DataFrame(results)
    return results_df

# --- วิธีการใช้งาน ---
if __name__ == "__main__":
    # ใช้ WDC แทน SNDK เนื่องจาก SNDK ถูกควบรวมและไม่มีการซื้อขายใน Ticker เดิมแล้ว
    stock_list = ['SNDK', 'COHR', 'LITE', 'OXY', 'COP'] 
    
    df_result = calculate_rvol_30m(stock_list)
    
    if not df_result.empty:
        print("\n==================== ผลลัพธ์ RVOL (30m) ====================")
        print(df_result.to_string(index=False))

กำลังดึงข้อมูลและคำนวณของหุ้น: SNDK...
กำลังดึงข้อมูลและคำนวณของหุ้น: COHR...
กำลังดึงข้อมูลและคำนวณของหุ้น: LITE...
กำลังดึงข้อมูลและคำนวณของหุ้น: OXY...
กำลังดึงข้อมูลและคำนวณของหุ้น: COP...

==================== ผลลัพธ์ RVOL (30m) ====================
Ticker       Date Volume_Today (30m) Avg_Volume (30m)  RVOL                        Signal
  SNDK 2026-03-04          9,396,187        8,201,999  1.15      Normal / Sideways (ปกติ)
  COHR 2026-03-04          4,525,286        2,262,176  2.00 Strong Reversal (กลับตัวจริง)
  LITE 2026-03-04          3,128,126        1,606,866  1.95 Strong Reversal (กลับตัวจริง)
   OXY 2026-03-04          5,929,352        6,020,006  0.98      Normal / Sideways (ปกติ)
   COP 2026-03-04          2,480,072        2,200,328  1.13      Normal / Sideways (ปกติ)


วิเคราะห์ Real Bearish Reversal

In [9]:
import yfinance as yf
import pandas as pd
import numpy as np

def check_pullback_or_bearish(tickers):
    results = []
    
    for ticker in tickers:
        try:
            # 1. ดึงข้อมูลรายวันย้อนหลัง 6 เดือน
            df = yf.download(ticker, period='6mo', interval='1d', progress=False)
            if df.empty:
                continue
                
            # รองรับ yfinance รูปแบบใหม่ที่เป็น MultiIndex
            if isinstance(df.columns, pd.MultiIndex):
                close = df['Close', ticker].dropna()
                high = df['High', ticker].dropna()
                low = df['Low', ticker].dropna()
                vol = df['Volume', ticker].dropna()
            else:
                close = df['Close'].dropna()
                high = df['High'].dropna()
                low = df['Low'].dropna()
                vol = df['Volume'].dropna()

            current_price = float(close.iloc[-1])
            
            # --- 2. คำนวณ Indicators ---
            # 2.1 Trend: ขาขึ้นระยะกลางยังอยู่ไหม? (EMA 50)
            ema_50 = close.ewm(span=50, adjust=False).mean()
            current_ema50 = float(ema_50.iloc[-1])
            is_above_ema50 = current_price > current_ema50
            
            # 2.2 Fibonacci 61.8% (Golden Ratio): โครงสร้างราคาเสียหรือยัง?
            recent_high = float(high.tail(45).max()) # High สูงสุดใน 1.5 เดือน
            recent_low = float(low.tail(60).min())   # Low ต่ำสุดใน 2 เดือนเพื่อเป็นฐาน
            
            # คำนวณระยะ Retracement
            fib_618 = recent_high - (0.618 * (recent_high - recent_low))
            is_above_fib618 = current_price > fib_618
            
            # 2.3 Volume Analysis: การขายครั้งนี้มีนัยสำคัญแค่ไหน?
            avg_vol_20 = float(vol.tail(20).mean())
            
            # ดู Volume เฉลี่ยเฉพาะ "วันที่ราคาปิดลบ" ในรอบ 5 วันล่าสุด
            recent_5_days_return = close.pct_change().tail(5)
            down_days_vol = vol.tail(5)[recent_5_days_return < 0]
            
            if len(down_days_vol) > 0:
                avg_down_vol = float(down_days_vol.mean())
                # ถ้าร่วงด้วย Volume ที่สูงกว่าค่าเฉลี่ย 20 วัน ถือว่าอันตราย (Distribution)
                is_high_sell_vol = avg_down_vol > avg_vol_20
            else:
                is_high_sell_vol = False
                
            # --- 3. ประมวลผลลัพธ์ (Decision Logic) ---
            score = 0
            if is_above_ema50: score += 1
            if is_above_fib618: score += 1
            if not is_high_sell_vol: score += 1 # ได้คะแนนถ้าลงด้วยวอลุ่มน้อย (แห้ง)
            
            if score == 3:
                status = "🟢 Pullback (ย่อเพื่อไปต่อ โครงสร้างแข็งแกร่ง)"
            elif score == 2:
                status = "🟡 Warning (เริ่มเสี่ยง ต้องลุ้นที่แนวรับ)"
            elif score == 1:
                status = "🟠 High Risk (อันตราย โครงสร้างเริ่มพัง)"
            else:
                status = "🔴 Bearish Reversal (เปลี่ยนเทรนเป็นลงชัดเจน)"
                
            results.append({
                'Ticker': ticker,
                'Price': f"{current_price:.2f}",
                'EMA_50': f"{current_ema50:.2f}",
                'Fib_61.8%': f"{fib_618:.2f}",
                'High_Sell_Vol': "Yes" if is_high_sell_vol else "No",
                'Status': status
            })
            
        except Exception as e:
            print(f"Error processing {ticker}: {e}")

    return pd.DataFrame(results)

# --- วิธีรัน ---
if __name__ == "__main__":
    # ใส่ชื่อหุ้นที่ต้องการประเมิน
    stock_list = ['OXY', 'COP', 'SNDK', 'COHR', 'LITE']
    
    df_result = check_pullback_or_bearish(stock_list)
    print("\n==================== Pullback vs Reversal Analysis ====================")
    print(df_result.to_string(index=False))


==================== Pullback vs Reversal Analysis ====================
Ticker  Price EMA_50 Fib_61.8% High_Sell_Vol                                        Status
   OXY  53.63  46.76     45.50           Yes     🟡 Warning (เริ่มเสี่ยง ต้องลุ้นที่แนวรับ)
   COP 115.61 104.10    102.45            No 🟢 Pullback (ย่อเพื่อไปต่อ โครงสร้างแข็งแกร่ง)
  SNDK 598.97 501.20    400.24            No 🟢 Pullback (ย่อเพื่อไปต่อ โครงสร้างแข็งแกร่ง)
  COHR 266.50 218.87    218.19            No 🟢 Pullback (ย่อเพื่อไปต่อ โครงสร้างแข็งแกร่ง)
  LITE 656.60 504.78    490.42            No 🟢 Pullback (ย่อเพื่อไปต่อ โครงสร้างแข็งแกร่ง)
